In [ ]:
!pip install pennylane pennylane-lightning -q

# ⚛️ Kuantum Temelleri — II

**Dr. Buket Toptaş** | Kuantum Makine Öğrenmesi Dersi

**Konular:** Bloch Küresi · Süperpozisyon · Ölçüm ve Born Kuralı · Dolanıklık

---
## 1. Bloch Küresi

Tek qubit'in durumu 3D küre üzerinde bir nokta:

$$|\psi\rangle = \cos\frac{\theta}{2}|0\rangle + e^{i\phi}\sin\frac{\theta}{2}|1\rangle$$

- **Kuzey kutbu** (θ=0): |0⟩
- **Güney kutbu** (θ=π): |1⟩
- **Ekvator** (θ=π/2): Süperpozisyon durumları

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(9, 8))
ax = fig.add_subplot(111, projection='3d')

# Küre
u = np.linspace(0, 2*np.pi, 40)
v = np.linspace(0, np.pi, 30)
x = np.outer(np.cos(u), np.sin(v))
y = np.outer(np.sin(u), np.sin(v))
z = np.outer(np.ones_like(u), np.cos(v))
ax.plot_wireframe(x, y, z, color='#94a3b8', alpha=0.08, lw=0.5)

# Eksenler
for vec, col in [((0,0,1.4),'#3b82f6'), ((0,0,-1.4),'#ef4444'), 
                  ((1.4,0,0),'#94a3b8'), ((0,1.4,0),'#94a3b8')]:
    ax.quiver(0,0,0, *vec, color=col, arrow_length_ratio=0.06, lw=1.5 if col!='#94a3b8' else 1, alpha=0.5 if col=='#94a3b8' else 1)

# Önemli durumlar
states = {'|0⟩':(0,0,1,'#3b82f6'), '|1⟩':(0,0,-1,'#ef4444'), '|+⟩':(1,0,0,'#10b981'),
          '|−⟩':(-1,0,0,'#f59e0b'), '|i⟩':(0,1,0,'#7c3aed'), '|−i⟩':(0,-1,0,'#d946ef')}

for name, (sx,sy,sz,c) in states.items():
    ax.scatter(sx, sy, sz, s=120, color=c, zorder=5, edgecolors='white', lw=1.5)
    ax.text(sx*1.3, sy*1.3, sz*1.2, name, fontsize=13, fontweight='bold', color=c, ha='center')

# Örnek |ψ⟩
th, ph = np.pi/3, np.pi/4
sx = np.sin(th)*np.cos(ph); sy = np.sin(th)*np.sin(ph); sz = np.cos(th)
ax.quiver(0,0,0, sx,sy,sz, color='#f59e0b', arrow_length_ratio=0.1, lw=3)
ax.text(sx*1.2, sy*1.2, sz*1.1, '|ψ⟩', fontsize=14, fontweight='bold', color='#f59e0b')

ax.set_xlim(-1.5,1.5); ax.set_ylim(-1.5,1.5); ax.set_zlim(-1.5,1.5)
ax.set_title('Bloch Küresi', fontsize=14, fontweight='bold')
ax.view_init(elev=20, azim=35)
plt.tight_layout()
plt.show()

---
## 2. Süperpozisyon

Bir qubit aynı anda **hem 0 hem 1** olabilir: $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$

Bu "kararsızlık" değil, gerçek bir fiziksel durumdur. Ölçene kadar qubit her iki durumda birden var.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))

states = [
    ('|0⟩', 1, 0),
    ('|+⟩', 1/np.sqrt(2), 1/np.sqrt(2)),
    ('0.95|0⟩+0.31|1⟩', 0.95, 0.31),
    ('|1⟩', 0, 1),
]

for ax, (name, a, b) in zip(axes, states):
    probs = [abs(a)**2, abs(b)**2]
    ax.bar(['|0⟩','|1⟩'], probs, color=['#3b82f6','#ef4444'], alpha=0.85, width=0.5, edgecolor='white')
    ax.set_ylim(0, 1.15)
    ax.set_title(name, fontsize=12, fontweight='bold')
    for j, p in enumerate(probs):
        if p > 0.01:
            ax.text(j, p+0.04, f'%{p*100:.0f}', ha='center', fontsize=12, fontweight='bold')
    ax.axhline(y=0.5, color='gray', ls='--', alpha=0.2)

plt.suptitle('Süperpozisyon — Farklı Durumların Olasılık Dağılımları', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. Ölçüm ve Born Kuralı

Ölçüm süperpozisyonu **çökertir**. Sonuç kesin: 0 veya 1. Geri dönüşü yok!

$$P(0) = |\alpha|^2, \quad P(1) = |\beta|^2 \quad \text{(Born kuralı)}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
alpha = np.cos(np.pi/6)   # cos(30°) ≈ 0.87
beta  = np.sin(np.pi/6)   # sin(30°) = 0.50

# 1000 ölçüm simülasyonu
n = 1000
results = np.random.choice([0, 1], size=n, p=[abs(alpha)**2, abs(beta)**2])
c0, c1 = np.sum(results==0), np.sum(results==1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

# Sol: Durum
ax1.axis('off')
ax1.text(0.5, 0.85, f'|ψ⟩ = {alpha:.2f}|0⟩ + {beta:.2f}|1⟩', fontsize=16, ha='center',
         fontfamily='monospace', transform=ax1.transAxes,
         bbox=dict(boxstyle='round', facecolor='#1e293b', edgecolor='#3b82f6', lw=2), color='white')
ax1.annotate('', xy=(0.5,0.55), xytext=(0.5,0.72), xycoords='axes fraction',
             arrowprops=dict(arrowstyle='->', color='#f59e0b', lw=3))
ax1.text(0.5, 0.47, '⬇️ ÖLÇÜM', fontsize=13, ha='center', transform=ax1.transAxes, color='#f59e0b', fontweight='bold')
ax1.text(0.5, 0.25, f'P(0) = |{alpha:.2f}|² = %{abs(alpha)**2*100:.0f}', fontsize=14, ha='center',
         transform=ax1.transAxes, color='#3b82f6', fontweight='bold')
ax1.text(0.5, 0.10, f'P(1) = |{beta:.2f}|² = %{abs(beta)**2*100:.0f}', fontsize=14, ha='center',
         transform=ax1.transAxes, color='#ef4444', fontweight='bold')

# Sağ: Histogram
bars = ax2.bar(['|0⟩','|1⟩'], [c0,c1], color=['#3b82f6','#ef4444'], alpha=0.85, width=0.5, edgecolor='white')
ax2.set_title(f'{n} Ölçüm Sonucu', fontsize=13, fontweight='bold')
for b, c in zip(bars, [c0,c1]):
    ax2.text(b.get_x()+b.get_width()/2, c+15, f'{c} (%{c/10:.0f})', ha='center', fontsize=12, fontweight='bold')
ax2.set_ylabel('Sayı')

plt.tight_layout()
plt.show()

---
## 4. Dolanıklık (Entanglement)

İki qubit birbirine bağlanır: birini ölçünce diğerinin durumunu **anında** bilirsin.

**Bell durumu:** $|\Phi^+\rangle = \frac{|00\rangle + |11\rangle}{\sqrt{2}}$

"Ya ikisi de 0, ya ikisi de 1. Başka ihtimal yok." Einstein buna **"ürkütücü uzaktan etki"** dedi.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
n = 1000
outcomes = np.random.choice(['00','01','10','11'], size=n, p=[0.5, 0, 0, 0.5])
counts = {s: np.sum(outcomes==s) for s in ['00','01','10','11']}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

# Histogram
colors = ['#3b82f6','#94a3b8','#94a3b8','#ef4444']
bars = ax1.bar(counts.keys(), counts.values(), color=colors, alpha=0.85, width=0.5, edgecolor='white')
ax1.set_title('Bell Durumu: (|00⟩+|11⟩)/√2', fontsize=13, fontweight='bold')
ax1.set_ylabel(f'{n} ölçüm')
for b, (k,v) in zip(bars, counts.items()):
    ax1.text(b.get_x()+b.get_width()/2, v+15, str(v), ha='center', fontsize=12, fontweight='bold')

# Korelasyon tablosu
ax2.axis('off')
ax2.set_title('Dolanıklık Korelasyonu', fontsize=13, fontweight='bold')
data = [['Qubit A','Qubit B','Olasılık'], ['0','0','%50 ✅'], ['1','1','%50 ✅'], ['0','1','%0 ❌'], ['1','0','%0 ❌']]
cols = [['#475569']*3, ['#10b981']*3, ['#10b981']*3, ['#ef4444']*3, ['#ef4444']*3]
t = ax2.table(cellText=data, cellColours=cols, cellLoc='center', loc='center')
t.auto_set_font_size(False); t.set_fontsize(13); t.scale(1, 2.2)
for cell in t.get_celld().values():
    cell.set_text_props(color='white', fontweight='bold')
    cell.set_edgecolor('#1e293b')

plt.tight_layout()
plt.show()

In [ ]:
import pennylane as qml

dev = qml.device('default.qubit', wires=2, shots=1000)

@qml.qnode(dev)
def bell_circuit():
    qml.Hadamard(wires=0)       # Süperpozisyon
    qml.CNOT(wires=[0, 1])      # Dolanıklık
    return qml.counts()

print("=== Bell Devresi ===")
print(qml.draw(bell_circuit)())
print()

results = bell_circuit()
print("=== Sonuçlar (1000 shot) ===")
for state, count in sorted(results.items()):
    print(f"  |{state}⟩ → {count} kez (%{count/10:.0f})")

print()
print("→ Sadece |00⟩ ve |11⟩ çıkıyor = DOLANIKLIK!")